# 0. Importaciones y configuración

In [ ]:
import json
import os
import random
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision.utils import save_image

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)

### Rutas

In [ ]:
RUTA_DATASET = Path("data")             # debe contener las carpetas Holograms y Phase
RUTA_SALIDA = Path("resultados")                        # muestras, checkpoints y curvas, una subcarpeta por experimento
RUTA_SPLIT = Path("splits/split.csv")      # partición train/val/test por holograma original

RUTA_SALIDA.mkdir(parents=True, exist_ok=True)
print("Dataset:", RUTA_DATASET.resolve())
print("Salida :", RUTA_SALIDA.resolve())

### Configuración del experimento

Todo lo que cambia entre experimentos está en `CFG`, que además se guarda como `config.json` en la carpeta de cada experimento. 

In [ ]:
RADIO_CENTRO = 15   # px que se ignoran alrededor del pico central al buscar el pico lateral.
                    # SUPOSICIÓN: se verifica visualmente en la sección 3.

CFG = dict(
    nombre="pix2pix_base",
    semilla=0,
    batch=8,
    ngf=64,                 # canales base del generador
    ndf=64,                 # canales base del discriminador
    lr_G=2e-4,              # tasa de aprendizaje del generador
    lr_D=2e-4,              # tasa de aprendizaje del discriminador
    beta1=0.5,              # parámetro beta1 del optimizador Adam
    lambda_l1=100.0,        # peso de la pérdida L1
    n_val_epoca=256,        # cuántas muestras de validación por época
    guardar_cada=5,         # cada cuántas épocas guardar el generador
)

---
# 1. Funciones

In [ ]:
PATRON_NOMBRE = re.compile(
    r"Image__(\d{4}-\d{2}-\d{2})__(\d{2}-\d{2}-\d{2})_(\d+)-(\d+)(?:\.png)?(?:-([A-Za-z]))?(?:\.png)+"
)


def leer_png(ruta):
    """Imagen 2D en [0, 1]. Soporta PNG de 8 y de 16 bits."""
    img = Image.open(ruta)
    if img.mode in ("I;16", "I;16B", "I"):
        return np.array(img, dtype=np.float32) / 65535.0
    return np.array(img.convert("L"), dtype=np.float32) / 255.0


def indexar_carpeta(ruta_base, carpetas=None):
    """Tabla con una fila por imagen. La letra final del nombre es opcional ('orig' si no hay)."""
    ruta_base = Path(ruta_base)
    if carpetas is None:
        carpetas = sorted(p.name for p in ruta_base.iterdir() if p.is_dir())
    filas, otros = [], []
    for carpeta in carpetas:
        for ruta in sorted((ruta_base / carpeta).glob("*.png")):
            m = PATRON_NOMBRE.fullmatch(ruta.name)
            if m is None:
                otros.append(ruta.name)
                continue
            fecha, hora, a, b, letra = m.groups()
            filas.append(dict(carpeta=carpeta, archivo=ruta.name, ruta=ruta,
                              original=f"{fecha}__{hora}", a=int(a), b=int(b),
                              variante=letra or "orig"))
    if otros:
        print(f"{ruta_base.name}: {len(otros)} archivos con otro formato de nombre (primeros 5): {otros[:5]}")
    return pd.DataFrame(filas)


def emparejar(df_holo, df_fase):
    """Une holograma y fase por (original, a, b, variante) y reporta lo que no coincide."""
    if df_holo.empty or df_fase.empty:
        raise ValueError("No se reconoció ningún nombre de archivo en Holograms o en Phase. "
                         "Revisar el patrón de nombres.")
    claves = ["original", "a", "b", "variante"]
    union = df_holo.merge(df_fase, on=claves, how="outer", suffixes=("_holo", "_fase"), indicator=True)
    n_ok = int((union["_merge"] == "both").sum())
    print(f"Pares completos: {n_ok}")
    print(f"Solo holograma (sin fase): {int((union['_merge'] == 'left_only').sum())} | "
          f"Solo fase (sin holograma): {int((union['_merge'] == 'right_only').sum())}")
    pares = union[union["_merge"] == "both"].drop(columns="_merge").copy()
    print("Mismo nombre de archivo en ambas carpetas:", f"{100 * (pares['archivo_holo'] == pares['archivo_fase']).mean():.1f} %")
    print("Misma subcarpeta en ambas:", f"{100 * (pares['carpeta_holo'] == pares['carpeta_fase']).mean():.1f} %")
    pares = pares.rename(columns={"carpeta_holo": "carpeta", "archivo_holo": "archivo"})
    return pares.reset_index(drop=True)


def parches_en_ambas_carpetas(df):
    """Claves (original, a, b) con versiones en más de una carpeta, y el total de recortes."""
    n = df.groupby(["original", "a", "b"])["carpeta"].nunique()
    return n[n > 1].index, len(n)


def mostrar_ejemplo_fuga(df, indice=0):
    """Un recorte con versiones en dos carpetas distintas, una al lado de la otra."""
    mixtos, _ = parches_en_ambas_carpetas(df)
    if len(mixtos) == 0:
        print("No hay recortes repartidos entre carpetas.")
        return
    original, a, b = mixtos[indice]
    parche = df[(df.original == original) & (df.a == a) & (df.b == b)]
    f1 = parche.iloc[0]
    f2 = parche[parche.carpeta != f1.carpeta].iloc[0]

    fig, ejes = plt.subplots(1, 2, figsize=(10, 5))
    for eje, f in zip(ejes, (f1, f2)):
        eje.imshow(leer_png(f.ruta_holo), cmap="gray")
        eje.set_title(f"{f.carpeta}\n{f.archivo}", fontsize=8)
        eje.axis("off")
    plt.tight_layout()
    plt.show()


def partir_por_original(df, proporciones=(0.70, 0.15, 0.15), semilla=42, ruta_csv=None):
    """Reparte los hologramas ORIGINALES (no las imágenes) entre train, val y test.
    Todas las imágenes de un mismo original quedan en el mismo conjunto.
    Si ruta_csv existe, se carga y se usa tal cual; si no, se crea y se guarda."""
    if ruta_csv is not None and Path(ruta_csv).exists():
        tabla = pd.read_csv(ruta_csv)
        print("Partición cargada de", ruta_csv)
    else:
        originales = np.array(sorted(df["original"].unique()))
        np.random.default_rng(semilla).shuffle(originales)
        n = len(originales)
        n_val, n_test = round(proporciones[1] * n), round(proporciones[2] * n)
        n_train = n - n_val - n_test
        tabla = pd.DataFrame({"original": originales,
                              "particion": ["train"] * n_train + ["val"] * n_val + ["test"] * n_test})
        if ruta_csv is not None:
            Path(ruta_csv).parent.mkdir(parents=True, exist_ok=True)
            tabla.sort_values("original").to_csv(ruta_csv, index=False)
            print("Partición creada y guardada en", ruta_csv)
    df = df.merge(tabla, on="original", how="left")
    assert df["particion"].notna().all(), "Hay hologramas originales sin partición asignada"
    return df


class ParesDataset(Dataset):
    """Entrega (fase, holograma), ambos de forma (1, 256, 256) y en [-1, 1]."""
    def __init__(self, rutas_fase, rutas_holo):
        assert len(rutas_fase) == len(rutas_holo)
        self.rutas_fase = list(rutas_fase)
        self.rutas_holo = list(rutas_holo)

    def __len__(self):
        return len(self.rutas_fase)

    def __getitem__(self, i):
        fase = leer_png(self.rutas_fase[i]) * 2 - 1
        holo = leer_png(self.rutas_holo[i]) * 2 - 1
        return torch.from_numpy(fase).unsqueeze(0), torch.from_numpy(holo).unsqueeze(0)


def crear_loader(df, cfg, entrenamiento=True):
    """DataLoader de pares para las filas de df (ya filtradas por partición)."""
    ds = ParesDataset(df["ruta_fase"].tolist(), df["ruta_holo"].tolist())
    return DataLoader(ds, batch_size=cfg["batch"], shuffle=entrenamiento, num_workers=0, drop_last=entrenamiento)


def cargar_pares(df, n, semilla=0):
    """n pares equiespaciados de df como tensores (n, 1, 256, 256): (fases, hologramas)."""
    idx = np.linspace(0, len(df) - 1, min(n, len(df))).astype(int)
    sub = df.iloc[idx]
    ds = ParesDataset(sub["ruta_fase"].tolist(), sub["ruta_holo"].tolist())
    fases, holos = zip(*[ds[i] for i in range(len(ds))])
    return torch.stack(fases), torch.stack(holos)


def mostrar_pares(df, indices):
    """Fase y holograma lado a lado, para comprobar a ojo que corresponden."""
    fig, ejes = plt.subplots(2, len(indices), figsize=(4 * len(indices), 8))
    ejes = np.asarray(ejes).reshape(2, -1)
    for j, i in enumerate(indices):
        f = df.iloc[i]
        ejes[0, j].imshow(leer_png(f.ruta_fase), cmap="gray")
        ejes[0, j].set_title("Fase\n" + f.archivo_fase, fontsize=7)
        ejes[1, j].imshow(leer_png(f.ruta_holo), cmap="gray")
        ejes[1, j].set_title("Holograma\n" + f.archivo, fontsize=7)
        ejes[0, j].axis("off"); ejes[1, j].axis("off")
    plt.tight_layout()
    plt.show()

## 1.2 Análisis de Fourier

Un holograma *off-axis* tiene en su transformada de Fourier un pico central y dos picos laterales. El pico lateral contiene la información de la fase.

- `metricas_picos`: busca el pico lateral más fuerte y devuelve su distancia al centro, su ángulo y su prominencia.
- `reconstruir_fase`: reconstruye la fase de un holograma con la FFT. Filtra alrededor del pico lateral, lo lleva al centro, quita la inclinación de fase residual y toma el ángulo de la transformada inversa.
- `comparar_fase`: compara la fase reconstruida con la fase de referencia .

**Cómo se compara la fase.** Los mapas de fase del dataset son PNG que salieron del método de Trujillo, no de una FFT simple. Por eso no se compara píxel a píxel. Se ajusta `fase_reconstruida ≈ a · fase_referencia + b + c·x + d·y` por mínimos cuadrados (escala, desfase y una inclinación residual, que son las diferencias esperables entre los dos métodos) y se mide lo que **no** explica ese ajuste:

- `r2`: fracción de la variación de la fase reconstruida que se explica con la referencia (1 = perfecto, 0 = nada).
- `nrmse`: error cuadrático medio del residuo, dividido por la variación de la referencia escalada (0 = perfecto).
- `escala`: el coeficiente `a`. Si es negativo, la fase reconstruida está invertida respecto a la referencia, lo que depende de qué pico lateral se use.

El resultado del holograma **real** es el piso: es lo mejor que se puede lograr con este método. El holograma generado se compara contra ese piso, no contra cero.

In [ ]:
def espectro_log(img):
    """Magnitud del espectro (log), con la frecuencia cero en el centro."""
    return np.log1p(np.abs(np.fft.fftshift(np.fft.fft2(img))))


def ver_fourier(imagen, titulo=None):
    """Muestra una imagen y su transformada de Fourier. Acepta ruta o arreglo 2D."""
    if isinstance(imagen, (str, Path)):
        imagen = leer_png(imagen)
    else:
        imagen = np.asarray(imagen, dtype=np.float32)
    fig, ejes = plt.subplots(1, 2, figsize=(10, 5))
    ejes[0].imshow(imagen, cmap="gray")
    ejes[0].set_title(titulo or "Imagen")
    ejes[0].axis("off")
    ejes[1].imshow(espectro_log(imagen), cmap="gray")
    ejes[1].set_title("Transformada de Fourier (log)")
    ejes[1].axis("off")
    plt.tight_layout()
    plt.show()


def metricas_picos(img, radio_centro=None):
    """Pico lateral más fuerte: distancia al centro, ángulo (mod 180) y prominencia."""
    radio_centro = RADIO_CENTRO if radio_centro is None else radio_centro
    img = img - img.mean()                                  # quita el término DC
    mag = np.abs(np.fft.fftshift(np.fft.fft2(img)))
    h, w = mag.shape
    cy, cx = h // 2, w // 2
    yy, xx = np.ogrid[:h, :w]
    dist = np.hypot(yy - cy, xx - cx)

    fuera = mag.copy()
    fuera[dist < radio_centro] = 0
    iy, ix = np.unravel_index(np.argmax(fuera), fuera.shape)
    fondo = np.median(mag[dist >= radio_centro])
    return {
        "dist": float(dist[iy, ix]),
        "angulo": float(np.degrees(np.arctan2(cy - iy, ix - cx)) % 180),
        "prominencia": float(fuera[iy, ix] / fondo),
        "_pos": (int(iy), int(ix)),
    }


def tabla_picos(imagenes):
    """DataFrame con las métricas de picos de cada imagen."""
    return pd.DataFrame([metricas_picos(im) for im in imagenes]).drop(columns="_pos")


def verificar_radio(img, radio_centro=None):
    """Dibuja el radio ignorado y el pico detectado, para comprobar que RADIO_CENTRO es razonable."""
    radio_centro = RADIO_CENTRO if radio_centro is None else radio_centro
    m = metricas_picos(img, radio_centro)
    h, w = img.shape
    fig, eje = plt.subplots(figsize=(6, 6))
    eje.imshow(espectro_log(img), cmap="gray")
    eje.add_patch(plt.Circle((w // 2, h // 2), radio_centro, fill=False, color="cyan", lw=1.5))
    eje.plot(m["_pos"][1], m["_pos"][0], "r+", ms=18, mew=2)
    eje.set_title(f"dist={m['dist']:.1f}  prominencia={m['prominencia']:.0f}\n"
                  f"círculo = zona ignorada, + = pico detectado")
    eje.axis("off")
    plt.show()


def reconstruir_fase(holo, radio_filtro=None, radio_centro=None):
    """Fase envuelta (rad) de un holograma off-axis 2D, por filtrado del pico lateral.
    radio_filtro: radio del filtro en px (por defecto, un tercio de la distancia al pico)."""
    holo = np.asarray(holo, dtype=np.float64)
    h, w = holo.shape
    m = metricas_picos(holo, radio_centro)
    iy, ix = m["_pos"]
    cy, cx = h // 2, w // 2
    radio = radio_filtro if radio_filtro is not None else m["dist"] / 3
    F = np.fft.fftshift(np.fft.fft2(holo - holo.mean()))
    yy, xx = np.ogrid[:h, :w]
    F = np.where(np.hypot(yy - iy, xx - ix) <= radio, F, 0)
    F = np.roll(F, (cy - iy, cx - ix), axis=(0, 1))          # lleva el pico al centro
    campo = np.fft.ifft2(np.fft.ifftshift(F))

    # El pico casi nunca cae justo en un píxel: queda una inclinación de fase residual que,
    # al pasar de pi, se "enrolla" y estropea la comparación. Se estima con el gradiente
    # medio del campo complejo (robusto al enrollado) y se quita.
    gx = np.angle(np.sum(campo[:, 1:] * np.conj(campo[:, :-1])))
    gy = np.angle(np.sum(campo[1:, :] * np.conj(campo[:-1, :])))
    yy2, xx2 = np.mgrid[:h, :w]
    campo = campo * np.exp(-1j * (gx * xx2 + gy * yy2))
    return np.angle(campo)


def comparar_fase(fase_rec, fase_ref):
    """Ajusta fase_rec ~ a*fase_ref + b + c*x + d*y y mide lo que el ajuste no explica."""
    h, w = fase_ref.shape
    yy, xx = np.mgrid[:h, :w]
    A = np.column_stack([fase_ref.ravel(), np.ones(h * w), xx.ravel() / w, yy.ravel() / h])
    coef, *_ = np.linalg.lstsq(A, fase_rec.ravel(), rcond=None)
    resid = fase_rec.ravel() - A @ coef
    rec_sin_molestias = fase_rec.ravel() - A[:, 1:] @ coef[1:]    # sin desfase ni inclinación
    var_rec = rec_sin_molestias.var()
    escala_ref = np.abs(coef[0]) * fase_ref.ravel().std()
    return {
        "r2": float(1 - resid.var() / var_rec) if var_rec > 0 else np.nan,
        "nrmse": float(np.sqrt((resid ** 2).mean()) / escala_ref) if escala_ref > 0 else np.nan,
        "escala": float(coef[0]),
    }


def verificar_reconstruccion(holo, fase_ref):
    """Holograma, fase reconstruida con la FFT y fase de referencia, para inspección visual."""
    rec = reconstruir_fase(holo)
    met = comparar_fase(rec, fase_ref)
    fig, ejes = plt.subplots(1, 3, figsize=(15, 5))
    ejes[0].imshow(holo, cmap="gray"); ejes[0].set_title("Holograma")
    ejes[1].imshow(rec, cmap="gray"); ejes[1].set_title("Fase reconstruida con FFT")
    ejes[2].imshow(fase_ref, cmap="gray"); ejes[2].set_title(f"Fase de referencia\nr2={met['r2']:.2f}  nrmse={met['nrmse']:.2f}")
    for e in ejes:
        e.axis("off")
    plt.tight_layout()
    plt.show()
    return met

## 1.3 Modelos: pix2pix

Modelo tomado de la literatura: **pix2pix** (Isola, Zhu, Zhou y Efros, 2017), con la implementación de referencia `junyanz/pytorch-CycleGAN-and-pix2pix`. Se reescribe la arquitectura aquí para integrarla al notebook, y se conserva su estructura.

- **Generador: U-Net de 8 niveles.** Un codificador reduce la imagen de 256×256 a 1×1 y un decodificador la vuelve a ampliar. Las **conexiones de salto** pasan los detalles finos del codificador al decodificador. Esto importa aquí: las franjas del holograma son detalle fino que no debe perderse por el cuello de botella. Las primeras tres capas del decodificador interno llevan *dropout*.
- **Discriminador: PatchGAN de 70×70.** En vez de dar un solo veredicto por imagen, da una cuadrícula de 30×30 veredictos, cada uno sobre un parche de 70×70 píxeles. Recibe **fase y holograma concatenados** (2 canales), de modo que juzga si el holograma es coherente con esa fase. Al evaluar parches, se fija en la textura local (las franjas).
- **Pérdida:** adversaria + `lambda_l1` × L1 entre el holograma generado y el real. La L1 obliga al generador a acercarse al holograma correcto, y la adversaria a que sea nítido.

In [ ]:
class UnetSkip(nn.Module):
    """Un nivel de la U-Net: baja, llama al nivel interior, sube y concatena con la entrada."""
    def __init__(self, ext_nc, int_nc, sub=None, input_nc=None, outermost=False, innermost=False, dropout=False):
        super().__init__()
        self.outermost = outermost
        input_nc = input_nc or ext_nc
        down = nn.Conv2d(input_nc, int_nc, 4, 2, 1, bias=False)

        if outermost:
            up = nn.ConvTranspose2d(int_nc * 2, ext_nc, 4, 2, 1)
            modelo = [down, sub, nn.ReLU(True), up, nn.Tanh()]
        elif innermost:
            up = nn.ConvTranspose2d(int_nc, ext_nc, 4, 2, 1, bias=False)
            modelo = [nn.LeakyReLU(0.2, True), down, nn.ReLU(True), up, nn.BatchNorm2d(ext_nc)]
        else:
            up = nn.ConvTranspose2d(int_nc * 2, ext_nc, 4, 2, 1, bias=False)
            modelo = [nn.LeakyReLU(0.2, True), down, nn.BatchNorm2d(int_nc), sub,
                      nn.ReLU(True), up, nn.BatchNorm2d(ext_nc)]
            if dropout:
                modelo.append(nn.Dropout(0.5))
        self.modelo = nn.Sequential(*modelo)

    def forward(self, x):
        y = self.modelo(x)
        return y if self.outermost else torch.cat([x, y], 1)


class GeneradorUNet(nn.Module):
    """U-Net de 8 niveles para imágenes de 256x256: 256 -> 1 -> 256."""
    def __init__(self, in_nc=1, out_nc=1, ngf=64):
        super().__init__()
        bloque = UnetSkip(ngf * 8, ngf * 8, innermost=True)
        for _ in range(3):
            bloque = UnetSkip(ngf * 8, ngf * 8, sub=bloque, dropout=True)
        bloque = UnetSkip(ngf * 4, ngf * 8, sub=bloque)
        bloque = UnetSkip(ngf * 2, ngf * 4, sub=bloque)
        bloque = UnetSkip(ngf, ngf * 2, sub=bloque)
        self.modelo = UnetSkip(out_nc, ngf, sub=bloque, input_nc=in_nc, outermost=True)

    def forward(self, x):
        return self.modelo(x)


class DiscriminadorPatch(nn.Module):
    """PatchGAN 70x70. Entrada: fase y holograma concatenados (2 canales). Salida: cuadrícula 30x30."""
    def __init__(self, in_nc=2, ndf=64):
        super().__init__()

        def bloque(entrada, salida, stride):
            return [nn.Conv2d(entrada, salida, 4, stride, 1, bias=False),
                    nn.BatchNorm2d(salida), nn.LeakyReLU(0.2, True)]

        self.modelo = nn.Sequential(
            nn.Conv2d(in_nc, ndf, 4, 2, 1), nn.LeakyReLU(0.2, True),   # 256 -> 128
            *bloque(ndf, ndf * 2, 2),                                  # -> 64
            *bloque(ndf * 2, ndf * 4, 2),                              # -> 32
            *bloque(ndf * 4, ndf * 8, 1),                              # -> 31
            nn.Conv2d(ndf * 8, 1, 4, 1, 1),                            # -> 30x30
        )

    def forward(self, fase, holo):
        return self.modelo(torch.cat([fase, holo], dim=1))


def inicializar_pesos(m):
    """Inicialización del original: normal(0, 0.02)."""
    nombre = m.__class__.__name__
    if "Conv" in nombre:
        nn.init.normal_(m.weight, 0.0, 0.02)
    elif "BatchNorm" in nombre:
        nn.init.normal_(m.weight, 1.0, 0.02)
        nn.init.zeros_(m.bias)

## 1.4 Entrenamiento

En cada lote:

1. **Discriminador:** ve el par (fase, holograma real) con etiqueta 1 y el par (fase, holograma generado) con etiqueta 0.
2. **Generador:** busca que el discriminador etiquete su par como real y que su holograma se parezca al real.

Al final de **cada época**, sobre `n_val_epoca` pares de **validación**:

- `val_L1`: error L1 contra el holograma real. Junto con `loss_L1` de entrenamiento forman las **curvas de aprendizaje de entrenamiento y validación**.
- `val_prominencia`: prominencia del pico lateral de los hologramas generados.
- `val_nrmse_fase`: error de la fase reconstruida con FFT respecto a la fase de entrada (mediana de 32 pares).

Se guardan en cada época las muestras de 4 pares fijos (`epoca_XXX.png`: fila 1 = fases, fila 2 = generados, fila 3 = reales), el checkpoint completo `ultimo.pt` y el historial. El generador se guarda cada `guardar_cada` épocas y el de menor `val_L1` como `G_mejor.pt`.

Tamaños aproximados (estimados, no medidos): el generador tiene unos 54 millones de parámetros, unos 200 MB por archivo, y `ultimo.pt` unos 700 MB.

In [ ]:
CLAVES_HIST = ["loss_D", "loss_G_gan", "loss_L1", "D_real", "D_falsa",
               "val_L1", "val_prominencia", "val_nrmse_fase", "tiempo"]
crit_gan = nn.BCEWithLogitsLoss()
crit_l1 = nn.L1Loss()


@torch.no_grad()
def predecir(G, fases, batch=16):
    """Hologramas generados para un tensor de fases (n, 1, 256, 256). G en modo evaluación."""
    estado = G.training
    G.eval()
    salida = [G(fases[i:i + batch].to(device)).cpu() for i in range(0, len(fases), batch)]
    G.train(estado)
    return torch.cat(salida)


def metricas_val(exp):
    """L1, prominencia del pico y error de fase sobre el subconjunto fijo de validación."""
    fases, holos = exp["val_fase"], exp["val_holo"]
    pred = predecir(exp["G"], fases)
    val_l1 = float((pred - holos).abs().mean())
    pred_np = pred.squeeze(1).numpy()
    prom = float(tabla_picos(pred_np[:64])["prominencia"].mean())
    nrmse = []
    for i in range(min(32, len(pred_np))):
        rec = reconstruir_fase(pred_np[i])
        nrmse.append(comparar_fase(rec, fases[i, 0].numpy())["nrmse"])
    return val_l1, prom, float(np.nanmedian(nrmse))


def crear_experimento(cfg, df_val):
    """Modelos, optimizadores, historial y subconjunto fijo de validación."""
    torch.manual_seed(cfg["semilla"])
    np.random.seed(cfg["semilla"])
    random.seed(cfg["semilla"])

    carpeta = RUTA_SALIDA / cfg["nombre"]
    (carpeta / "muestras").mkdir(parents=True, exist_ok=True)
    (carpeta / "checkpoints").mkdir(parents=True, exist_ok=True)
    (carpeta / "config.json").write_text(json.dumps(cfg, indent=2))

    G = GeneradorUNet(1, 1, cfg["ngf"]).to(device)
    D = DiscriminadorPatch(2, cfg["ndf"]).to(device)
    G.apply(inicializar_pesos)
    D.apply(inicializar_pesos)

    val_fase, val_holo = cargar_pares(df_val, cfg["n_val_epoca"])
    return dict(
        cfg=cfg, carpeta=carpeta, G=G, D=D,
        opt_G=optim.Adam(G.parameters(), lr=cfg["lr_G"], betas=(cfg["beta1"], 0.999)),
        opt_D=optim.Adam(D.parameters(), lr=cfg["lr_D"], betas=(cfg["beta1"], 0.999)),
        val_fase=val_fase, val_holo=val_holo,
        hist={k: [] for k in CLAVES_HIST},
    )


def entrenar(exp, loader, epocas, max_lotes=None):
    """Entrena `epocas` más, continuando desde la última época registrada.
    max_lotes limita los lotes por época (solo para pruebas; usar un experimento aparte)."""
    cfg, G, D = exp["cfg"], exp["G"], exp["D"]
    opt_G, opt_D, hist, carpeta = exp["opt_G"], exp["opt_D"], exp["hist"], exp["carpeta"]
    inicio = len(hist["loss_D"])
    lam = cfg["lambda_l1"]

    for epoca in range(inicio, inicio + epocas):
        t0 = time.time()
        G.train(); D.train()
        acum, n = np.zeros(5), 0

        for i, (fase, holo) in enumerate(loader):
            if max_lotes is not None and i >= max_lotes:
                break
            fase, holo = fase.to(device), holo.to(device)
            falsa = G(fase)

            # ---- Discriminador ----
            opt_D.zero_grad()
            pred_real = D(fase, holo)
            pred_falsa = D(fase, falsa.detach())
            loss_D = 0.5 * (crit_gan(pred_real, torch.ones_like(pred_real)) +
                            crit_gan(pred_falsa, torch.zeros_like(pred_falsa)))
            loss_D.backward()
            opt_D.step()

            # ---- Generador ----
            opt_G.zero_grad()
            pred = D(fase, falsa)
            loss_gan = crit_gan(pred, torch.ones_like(pred))
            loss_l1 = crit_l1(falsa, holo)
            (loss_gan + lam * loss_l1).backward()
            opt_G.step()

            acum += [loss_D.item(), loss_gan.item(), loss_l1.item(),
                     torch.sigmoid(pred_real).mean().item(),
                     torch.sigmoid(pred_falsa).mean().item()]
            n += 1

        prom = acum / n
        for clave, valor in zip(["loss_D", "loss_G_gan", "loss_L1", "D_real", "D_falsa"], prom):
            hist[clave].append(float(valor))

        val_l1, val_prom, val_nrmse = metricas_val(exp)
        hist["val_L1"].append(val_l1)
        hist["val_prominencia"].append(val_prom)
        hist["val_nrmse_fase"].append(val_nrmse)
        hist["tiempo"].append(float(time.time() - t0))

        # Muestras de 4 pares fijos: fases, generados, reales
        fases4, reales4 = exp["val_fase"][:4], exp["val_holo"][:4]
        gen4 = predecir(G, fases4)
        save_image(torch.cat([fases4, gen4, reales4]), carpeta / "muestras" / f"epoca_{epoca + 1:03d}.png",
                   nrow=4, normalize=True, value_range=(-1, 1))

        # Generadores y checkpoint
        if val_l1 <= min(hist["val_L1"]):
            torch.save(G.state_dict(), carpeta / "checkpoints" / "G_mejor.pt")
        if (epoca + 1) % cfg["guardar_cada"] == 0:
            torch.save(G.state_dict(), carpeta / "checkpoints" / f"G_epoca_{epoca + 1:03d}.pt")
        tmp = carpeta / "checkpoints" / "ultimo_tmp.pt"
        torch.save({"epoca": epoca + 1, "G": G.state_dict(), "D": D.state_dict(),
                    "opt_G": opt_G.state_dict(), "opt_D": opt_D.state_dict(),
                    "hist": hist, "cfg": cfg}, tmp)
        os.replace(tmp, carpeta / "checkpoints" / "ultimo.pt")
        pd.DataFrame(hist).to_csv(carpeta / "historial.csv", index_label="epoca")

        print(f"Época {epoca + 1} | D: {prom[0]:.3f} | G_gan: {prom[1]:.3f} | L1 train: {prom[2]:.4f} | "
              f"L1 val: {val_l1:.4f} | D(real): {prom[3]:.2f} D(falsa): {prom[4]:.2f} | "
              f"prom. pico: {val_prom:.0f} | nrmse fase: {val_nrmse:.2f} | {hist['tiempo'][-1]:.0f} s")


def cargar_checkpoint(exp, ruta=None):
    """Retoma un experimento desde ultimo.pt (archivo propio, de confianza)."""
    ruta = ruta or exp["carpeta"] / "checkpoints" / "ultimo.pt"
    ck = torch.load(ruta, map_location=device, weights_only=False)
    exp["G"].load_state_dict(ck["G"])
    exp["D"].load_state_dict(ck["D"])
    exp["opt_G"].load_state_dict(ck["opt_G"])
    exp["opt_D"].load_state_dict(ck["opt_D"])
    exp["hist"] = ck["hist"]
    print(f"Retomado desde la época {ck['epoca']}")


def cargar_generador(exp, nombre_archivo="G_mejor.pt"):
    """Carga en exp['G'] un generador guardado (por ejemplo 'G_mejor.pt' o 'G_epoca_050.pt')."""
    ruta = exp["carpeta"] / "checkpoints" / nombre_archivo
    exp["G"].load_state_dict(torch.load(ruta, map_location=device))
    print("Generador cargado:", nombre_archivo)

## 1.5 Evaluación y gráficos

In [ ]:
def evaluar_conjunto(G, df, n=200):
    """Métricas por imagen sobre n pares de df. Devuelve (tabla, fases, reales, generados) en numpy."""
    fases, holos = cargar_pares(df, n)
    gens = predecir(G, fases)
    fases_np, holos_np, gens_np = (x.squeeze(1).numpy() for x in (fases, holos, gens))

    filas = []
    for fase, real, gen in zip(fases_np, holos_np, gens_np):
        mse = float(((gen - real) ** 2).mean())
        pr, pg = metricas_picos(real), metricas_picos(gen)
        fr = comparar_fase(reconstruir_fase(real), fase)
        fg = comparar_fase(reconstruir_fase(gen), fase)
        filas.append(dict(
            mae=float(np.abs(gen - real).mean()),
            psnr=10 * np.log10(4.0 / mse) if mse > 0 else np.inf,
            prom_real=pr["prominencia"], prom_gen=pg["prominencia"],
            dist_real=pr["dist"], dist_gen=pg["dist"],
            r2_real=fr["r2"], nrmse_real=fr["nrmse"],
            r2_gen=fg["r2"], nrmse_gen=fg["nrmse"],
        ))
    return pd.DataFrame(filas), fases_np, holos_np, gens_np


def resumen_evaluacion(tabla):
    """Media, desviación y mediana de cada métrica."""
    return tabla.agg(["mean", "std", "median"]).T.round(3)


def graficar_curvas(hist, titulo=""):
    """Curvas de aprendizaje (train y val), pérdidas GAN y métricas físicas."""
    ep = np.arange(1, len(hist["loss_D"]) + 1)
    fig, ejes = plt.subplots(2, 3, figsize=(16, 8))

    ejes[0, 0].plot(ep, hist["loss_L1"], label="L1 entrenamiento")
    ejes[0, 0].plot(ep, hist["val_L1"], label="L1 validación")
    ejes[0, 0].set_title("Curva de aprendizaje (L1)"); ejes[0, 0].legend()

    ejes[0, 1].plot(ep, hist["loss_D"], label="Discriminador")
    ejes[0, 1].plot(ep, hist["loss_G_gan"], label="Generador (término adversario)")
    ejes[0, 1].set_title("Pérdidas adversarias"); ejes[0, 1].legend()

    ejes[0, 2].plot(ep, hist["D_real"], label="D(real)")
    ejes[0, 2].plot(ep, hist["D_falsa"], label="D(falsa)")
    ejes[0, 2].axhline(0.5, color="gray", ls=":")
    ejes[0, 2].set_title("Probabilidad media asignada por D"); ejes[0, 2].legend()

    ejes[1, 0].plot(ep, hist["val_prominencia"], color="green")
    ejes[1, 0].set_title("Prominencia del pico lateral (validación)")

    ejes[1, 1].plot(ep, hist["val_nrmse_fase"], color="purple")
    ejes[1, 1].set_title("Error de fase reconstruida, nrmse (validación)")

    ejes[1, 2].plot(ep, hist["tiempo"], color="gray")
    ejes[1, 2].set_title("Segundos por época")

    for e in ejes.ravel():
        e.set_xlabel("época")
    fig.suptitle(titulo)
    plt.tight_layout()
    plt.show()


def ver_muestras(exp, epocas):
    """Cuadrículas de muestras (fases / generados / reales) de varias épocas."""
    fig, ejes = plt.subplots(len(epocas), 1, figsize=(10, 7.5 * len(epocas)))
    ejes = np.atleast_1d(ejes)
    for eje, e in zip(ejes, epocas):
        eje.imshow(Image.open(exp["carpeta"] / "muestras" / f"epoca_{e:03d}.png").convert("L"), cmap="gray")
        eje.set_title(f"Época {e} (arriba: fase | medio: generado | abajo: real)")
        eje.axis("off")
    plt.tight_layout()
    plt.show()


def comparar_visual(fases, reales, generados, indices):
    """Por cada índice: fase, holograma real, generado, diferencia y espectros de real y generado."""
    fig, ejes = plt.subplots(len(indices), 6, figsize=(20, 3.4 * len(indices)))
    ejes = np.asarray(ejes).reshape(len(indices), 6)
    for fila, i in zip(ejes, indices):
        fila[0].imshow(fases[i], cmap="gray");     fila[0].set_title("Fase (entrada)")
        fila[1].imshow(reales[i], cmap="gray");    fila[1].set_title("Holograma real")
        fila[2].imshow(generados[i], cmap="gray"); fila[2].set_title("Holograma generado")
        fila[3].imshow(np.abs(generados[i] - reales[i]), cmap="magma"); fila[3].set_title("|generado - real|")
        fila[4].imshow(espectro_log(reales[i]), cmap="gray");    fila[4].set_title("Fourier real")
        fila[5].imshow(espectro_log(generados[i]), cmap="gray"); fila[5].set_title("Fourier generado")
        for e in fila:
            e.axis("off")
    plt.tight_layout()
    plt.show()

---
# 2. Datos: pares, verificación, fuga de datos y partición

## 2.1 Estructura y emparejamiento

Cada nombre, por ejemplo `Image__2021-01-29__13-37-20_3-12-A.png`, contiene:

- `2021-01-29__13-37-20`: fecha y hora de captura, que identifica el **holograma original**.
- `3-12`: posición del **recorte** dentro del original.
- `A`, `B` o nada: **versión** del recorte (original, o rotada/volteada).

El dataset descargado tiene **192 originales × 64 recortes × 3 versiones = 36.864 pares**, distinto de lo que describe el artículo (300 originales y 28 recortes). Cada holograma se une con su fase por (original, recorte, versión).

In [ ]:
RUTA_HOLO = RUTA_DATASET / "Holograms"
RUTA_FASE = RUTA_DATASET / "Phase"
print("Subcarpetas de Holograms:", sorted(p.name for p in RUTA_HOLO.iterdir() if p.is_dir()))
print("Subcarpetas de Phase    :", sorted(p.name for p in RUTA_FASE.iterdir() if p.is_dir()))

df_holo = indexar_carpeta(RUTA_HOLO)
df_fase = indexar_carpeta(RUTA_FASE)
print("\nHologramas indexados:", len(df_holo), "| Fases indexadas:", len(df_fase))

df = emparejar(df_holo, df_fase)

print("\nPares por carpeta:\n", df.groupby("carpeta").size().to_string())
print("\nPares por versión:\n", df["variante"].value_counts().to_string())
print("\nHologramas originales distintos:", df["original"].nunique())
print("Recortes distintos (original, a, b):", df.groupby(["original", "a", "b"]).ngroups)
print("Total de pares:", len(df))       # esperado: 36864

### Verificación visual de la correspondencia

Que los nombres coincidan no basta: hay que confirmar que cada fase corresponde a su holograma (mismas células, en las mismas posiciones). Se muestran cuatro pares al azar. Cada columna es un par: arriba la fase y abajo su holograma.

In [ ]:
mostrar_pares(df, indices=np.random.default_rng(0).choice(len(df), 4, replace=False))

## 2.2 Diagnóstico: ¿la partición original es independiente?

El artículo reparte el dataset en 80 % y 20 % de las imágenes (29.491 y 7.373), pero no describe con qué criterio decide en qué carpeta cae cada una. Como las tres versiones de un recorte son la misma imagen rotada o volteada, si se repartieran **una por una al azar**, muchas versiones de un mismo recorte quedarían en carpetas distintas. Eso es **fuga de datos**.

La celda mide cuántos recortes tienen versiones en las dos carpetas y lo compara con lo esperado de un reparto aleatorio imagen por imagen. Si el reparto hubiera sido por holograma original, el resultado sería 0.

**Resultado en nuestra corrida: 5.837 de 12.288 recortes (47,5 %)**, cercano a lo esperado de un reparto aleatorio. Es evidencia de que la carpeta `Validation` no es independiente de `Training`.

In [ ]:
mixtos, total = parches_en_ambas_carpetas(df)
print(f"Recortes con versiones en más de una carpeta: {len(mixtos)} de {total} ({100 * len(mixtos) / total:.1f} %)")

# Esperado si cada imagen se asignara al azar e independientemente a la carpeta menor con probabilidad p
p = df["carpeta"].value_counts(normalize=True).min()
esperado = 1 - (1 - p) ** 3 - p ** 3
print(f"Esperado con reparto aleatorio imagen por imagen (p={p:.3f}): {100 * esperado:.1f} %")

## 2.3 Ejemplo concreto de fuga

Un recorte con una versión en `Training` y otra en `Validation`. Es la misma imagen rotada o volteada.

In [ ]:
mostrar_ejemplo_fuga(df, indice=0)
# mostrar_ejemplo_fuga(df, indice=1)
# mostrar_ejemplo_fuga(df, indice=2)

## 2.4 Partición corregida: por holograma original

**Decisión:** repartir los 192 hologramas originales en `train` (70 %), `val` (15 %) y `test` (15 %). Todas las imágenes que vienen del mismo original, con todos sus recortes y versiones, quedan en el mismo conjunto, **y la misma partición se aplica a las fases**, porque se reparten los pares.

- **Reproducibilidad:** semilla fija y archivo `splits/split.csv` (192 filas).
- **Conjunto de prueba aparte:** el dataset original solo trae dos carpetas. `val` sirve para decidir durante el entrenamiento y `test` queda para la evaluación final.

In [ ]:
df = partir_por_original(df, proporciones=(0.70, 0.15, 0.15), semilla=42, ruta_csv=RUTA_SPLIT)

# Verificación 1: ningún holograma original está en dos conjuntos a la vez
o = {p: set(df.loc[df.particion == p, "original"]) for p in ("train", "val", "test")}
print("Originales compartidos -> train y val:", len(o["train"] & o["val"]),
      "| train y test:", len(o["train"] & o["test"]),
      "| val y test:", len(o["val"] & o["test"]))                      # esperado: 0 0 0

# Verificación 2: las versiones de cada recorte quedan siempre juntas
print("Máximo de conjuntos por recorte:",
      df.groupby(["original", "a", "b"])["particion"].nunique().max())   # esperado: 1

print("\nHologramas originales por conjunto:", {p: len(v) for p, v in o.items()})
print("Pares por conjunto:", df["particion"].value_counts().to_dict())

## 2.5 DataLoaders

In [ ]:
df_train = df[df.particion == "train"]
df_val = df[df.particion == "val"]
df_test = df[df.particion == "test"]

loader_train = crear_loader(df_train, CFG, entrenamiento=True)
loader_val = crear_loader(df_val, CFG, entrenamiento=False)

print("Pares -> train:", len(loader_train.dataset), "| val:", len(loader_val.dataset),
      "| test:", len(df_test))

fase, holo = next(iter(loader_train))
print("Lote de fases     :", tuple(fase.shape), "| rango:", round(fase.min().item(), 3), round(fase.max().item(), 3))
print("Lote de hologramas:", tuple(holo.shape), "| rango:", round(holo.min().item(), 3), round(holo.max().item(), 3))

---
# 3. Referencia: pares reales y piso de la reconstrucción de fase


## 3.1 Espectro de un holograma real y validación de `RADIO_CENTRO`

El círculo cian debe cubrir el pico central y el `+` rojo debe caer sobre uno de los picos laterales.

In [ ]:
fases_ref, holos_ref = cargar_pares(df_train, 200)
fases_ref_np, holos_ref_np = fases_ref.squeeze(1).numpy(), holos_ref.squeeze(1).numpy()
print(holos_ref_np.shape)

ver_fourier(holos_ref_np[0], "Holograma real")
verificar_radio(holos_ref_np[0])

## 3.2 Reconstrucción de fase con FFT sobre hologramas reales

Se reconstruye la fase de un holograma real y se compara con su fase del dataset.

In [ ]:
met = verificar_reconstruccion(holos_ref_np[0], fases_ref_np[0])
print(met)

## 3.3 Piso del error y control de correspondencia

Se calcula `r2` y `nrmse` sobre 200 pares reales (el **piso**), y se repite emparejando cada holograma con la fase de **otro** recorte (el **control**). Si la correspondencia es correcta y la reconstrucción funciona, el piso debe ser claramente mejor que el control. Esto también es una verificación cuantitativa de que los pares están bien emparejados.

In [ ]:
piso = pd.DataFrame([comparar_fase(reconstruir_fase(h), f) for h, f in zip(holos_ref_np, fases_ref_np)])
control = pd.DataFrame([comparar_fase(reconstruir_fase(h), fases_ref_np[(i + 7) % len(fases_ref_np)])
                        for i, h in enumerate(holos_ref_np)])

print("PISO (holograma real con SU fase)")
print(piso.agg(["mean", "median"]).round(3).T.to_string())
print("\nCONTROL (holograma real con la fase de OTRO recorte)")
print(control.agg(["mean", "median"]).round(3).T.to_string())

---
# 4. Creación del experimento

Crea el generador (U-Net), el discriminador (PatchGAN) y los optimizadores según `CFG`, y carga el subconjunto fijo de validación para las métricas de cada época. **Cambiar `CFG["nombre"]` antes de correrla en cada experimento nuevo**, porque si no se sobrescribe la carpeta del anterior. Si cambias `batch`, vuelve a correr también la sección 2.5.

In [ ]:
print(json.dumps(CFG, indent=2))

exp = crear_experimento(CFG, df_val)

print("\nParámetros G:", sum(p.numel() for p in exp["G"].parameters()))
print("Parámetros D:", sum(p.numel() for p in exp["D"].parameters()))
print("Salida en:", exp["carpeta"])

---
# 5. Entrenamiento

## 5.1 Prueba rápida

Dos épocas con 100 lotes, en un experimento aparte (`prueba`). Sirve para comprobar que no hay errores y para **medir el tiempo**: una época completa tiene 3216 lotes, así que el tiempo de 100 lotes multiplicado por 32.16 estima una época.

In [ ]:
exp_prueba = crear_experimento({**CFG, "nombre": "prueba"}, df_val)
entrenar(exp_prueba, loader_train, epocas=2, max_lotes=100)
print("Lotes por época completa:", len(loader_train))

## 5.2 Entrenamiento completo

`entrenar` continúa desde la última época registrada, así que **esta celda se puede correr varias veces seguidas** para entrenar por tandas. Entre tandas, mirar `resultados/<nombre>/muestras/` y las curvas de la sección 6.

In [ ]:
# cargar_checkpoint(exp)
entrenar(exp, loader_train, epocas=10)

---
# 6. Análisis de resultados

## 6.1 Curvas de entrenamiento

Las pérdidas de una GAN oscilan y por sí solas no indican si el modelo mejora. La curva de L1 (entrenamiento contra validación) permite hablar de sobreajuste, y la prominencia del pico y el error de fase miden la propiedad física de lo generado.

In [ ]:
graficar_curvas(exp["hist"], titulo=CFG["nombre"])

## 6.2 Evolución de las muestras

Cada imagen muestra las fases de entrada (arriba), los hologramas generados (medio) y los reales (abajo), con los mismos cuatro pares en todas las épocas.

In [ ]:
ver_muestras(exp, [1, 5, 10])

## 6.3 Elegir el mejor generador

Se guarda `G_mejor.pt` (menor L1 de validación) y un generador cada `guardar_cada` épocas. Se elige con validación, nunca con `test`, y se confirma visualmente que las franjas se vean, porque la L1 favorece imágenes suavizadas.

In [ ]:
d = pd.DataFrame(exp["hist"])
d.index = d.index + 1
print(d.sort_values("val_L1").head(5)[["loss_L1", "val_L1", "val_prominencia", "val_nrmse_fase", "D_real", "D_falsa"]].round(4))

# cargar_generador(exp, "G_mejor.pt")        # o "G_epoca_050.pt"

## 6.4 Evaluación en validación

200 pares de validación. `mae` y `psnr` miden el parecido píxel a píxel con el real (la L1 penaliza franjas desfasadas aunque sean igual de buenas). `prom` y `dist` comparan el pico lateral, es decir, la intensidad y la frecuencia de las franjas. `nrmse_gen` se lee frente a `nrmse_real` (el piso): el error atribuible al modelo es la diferencia.

In [ ]:
tabla_val, fases_v, reales_v, gens_v = evaluar_conjunto(exp["G"], df_val, n=200)
print(resumen_evaluacion(tabla_val).to_string())

## 6.5 Comparación visual con espectros

Por cada par: fase, holograma real, generado, diferencia y espectros de Fourier. Se buscan los tres picos en ambos espectros.

In [ ]:
comparar_visual(fases_v, reales_v, gens_v, indices=[0, 40, 80, 120])